In [0]:
from pyspark.sql import functions as F

# A new notebook has its own memory, so we recreate the variables
silver = spark.table("workspace.silver.transactions")

In [0]:
# How many accounts, and are there null or empty ids?
print("distinct accounts:", silver.select("account_id").distinct().count())
print("null or empty account_id:",
      silver.filter(F.col("account_id").isNull() | (F.trim("account_id") == "")).count())

In [0]:
dim_account = (
    silver
    .select("account_id")
    .distinct()                                            # one row per account
    .withColumn("account_key", F.xxhash64("account_id"))   # deterministic surrogate key
    .select("account_key", "account_id")
)

(
    dim_account.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_account")
)

In [0]:
gold_acc = spark.table("workspace.gold.dim_account")
gold_acc.orderBy("account_id").show(5, truncate=False)

# rows and distinct keys must match (expected: 50 and 50)
print("rows:", gold_acc.count(), "| distinct keys:", gold_acc.select("account_key").distinct().count())

# every silver account must exist in the dimension. Expected: 0
silver_acc = silver.select("account_id").distinct()
missing = silver_acc.join(
    gold_acc, silver_acc["account_id"] == gold_acc["account_id"], "left_anti"
).count()
print("accounts without dimension row:", missing)

In [0]:
#Criar e gravar a fato

fact_transactions = silver.select(
    "transaction_id",
    # Same formula used in dim_date, so the keys match
    F.date_format("transaction_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.xxhash64("account_id").alias("account_key"),    # same hash used in dim_account
    F.xxhash64("category").alias("category_key"),     # same hash used in dim_category
    "status",
    "currency",
    "amount",
)

(
    fact_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_transactions")
)

In [0]:
fact = spark.table("workspace.gold.fact_transactions")
fact.printSchema()
fact.show(5, truncate=False)

# 1) Grain: one row per transaction (expected: 500 and 500)
print("rows:", fact.count(), "| distinct ids:", fact.select("transaction_id").distinct().count())

# 2) Orphan keys: fact rows with no match in a dimension (expected: 0 each)
dim_date = spark.table("workspace.gold.dim_date")
dim_acc = spark.table("workspace.gold.dim_account")
dim_cat = spark.table("workspace.gold.dim_category")
print("orphan date_key:", fact.join(dim_date, "date_key", "left_anti").count())
print("orphan account_key:", fact.join(dim_acc, "account_key", "left_anti").count())
print("orphan category_key:", fact.join(dim_cat, "category_key", "left_anti").count())

# 3) Reconciliation: the total amount must be the same as in silver
print("fact total:", fact.agg(F.sum("amount")).first()[0],
      "| silver total:", silver.agg(F.sum("amount")).first()[0])